# Phase 1: confirm the core result (BitNet b1.58 2B4T, code, Kaggle 2×T4)

Gate **G1** (fixed in advance): counter with ternary gradients reaches **MBPP+ ≥ 35.5 as a mean of 3 seeds** and **≥ zero-shot + 3**; **merged LoRA ≤ zero-shot + 1**; the counter **beats stochastic rounding (DQT-style) by ≥ 1.5** on MBPP+.

The notebook runs in two sessions (set `SESSION` below; finished runs are skipped, so re-running is safe):

| Session | GPU 0 | GPU 1 | ≈ time |
|---|---|---|---|
| **A** (decides G1) | counter s2 → SR s0 → LoRA s0 → zero-shot | SR s1 → SR s2 → LoRA s1 → counter+leak 0.003 s1 | 7 h |
| **B** (leak study) | leak 0.003 s2 → leak 0.001 s0 → SR at 1/10 rate s0 | leak 0.001 s1 → leak 0.001 s2 → SR at 1/100 rate s0 | 5 h |

All counter and SR runs: fp16 matmuls, ternary gradients, vote scale 0.03, τ = 32, 250 steps, the same 2,000 decontaminated Magicoder examples. **SR** = DQT-style stochastic rounding straight onto the ternary grid: same normalized vote and same expected drift as the counter, but no memory. LoRA uses the fixed merge (no memory blow-up).

The last cells print every result with 3-seed means, check gate G1 automatically, and rescore HumanEval with the prompt's imports added (the HumanEval-drop diagnostic).

Settings: **GPU T4 ×2**, internet **on**, **Save Version → Save & Run All**.

In [ ]:
!pip install -q -U transformers datasets accelerate psutil evalplus

In [ ]:
%%writefile /kaggle/working/bitnet_cpu_finetune.py
#!/usr/bin/env python
"""Fine-tune a ternary BitNet model on CPU and compare update rules.

Arms (--arm)
  zeroshot    no training: accuracy of the ternary model as released
  bop         ternary Bop baseline: 16-bit moving average of the same normalized votes, threshold, no reset
  counter     int8 flip counter on every ternary weight (ours): 2 bytes of state per weight,
              the update is applied inside the backward pass so no gradient is ever stored
  lora        LoRA adapters + AdamW on a frozen ternary base; evaluated before AND after merging
              the adapters back into ternary weights (the deployable model)
  latent_sgd  plain full fine-tuning: fp32 master weights, STE, the counter's normalized step applied
              inside the backward pass (LOMO-style); the fp32 counterpart of the int8 counter
  latent_adamw  standard full fine-tuning with AdamW: only prints the memory it would need

Task (--task)
  rte    textual entailment (GLUE RTE), accuracy on the 277 validation examples (default)
  sst2   sentiment (GLUE SST-2); the instruction-tuned model already scores ~90% zero-shot
  agnews news topic, 4 classes
  text   your own text: --train_file / --val_file, perplexity

Setup
  pip install -U torch transformers datasets accelerate psutil
  Runs on CPU or GPU (--device auto). Computation is fp32 on both, so accuracy results are comparable;
  use --max_steps 20 --skip_eval on CPU to measure time per step and memory only.
  python bitnet_cpu_finetune.py --arm zeroshot --out runs/zeroshot
  python bitnet_cpu_finetune.py --arm counter  --out runs/counter
  python bitnet_cpu_finetune.py --arm lora     --out runs/lora
  python bitnet_cpu_finetune.py --arm latent_sgd --out runs/latent_sgd
Start with --train_n 64 --eval_n 64 to check everything runs and to read the time estimate.
"""
import argparse, json, math, os, random, re, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

TARGETS = ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
TASKS = {
    'sst2':   dict(labels=[' negative', ' positive']),
    'rte':    dict(labels=[' Yes', ' No']),
    'agnews': dict(labels=[' World', ' Sports', ' Business', ' Science']),
}


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument('--arm', required=True, choices=['zeroshot', 'counter', 'bop', 'sr', 'lora', 'latent_sgd', 'latent_adamw'])
    p.add_argument('--model', default='microsoft/bitnet-b1.58-2B-4T-bf16')
    p.add_argument('--out', required=True)
    p.add_argument('--task', default='rte', choices=['sst2', 'rte', 'agnews', 'text', 'code'])
    p.add_argument('--code_dev', type=int, default=200, help='code: held-out training examples for monitoring (never the benchmark)')
    p.add_argument('--code_eval', default='humaneval,mbpp', help='code: EvalPlus benchmarks to run at the end')
    p.add_argument('--code_eval_n', type=int, default=0, help='code: only generate for the first N problems, no scoring (quick checks)')
    p.add_argument('--gen_bs', type=int, default=16)
    p.add_argument('--max_new', type=int, default=512)
    p.add_argument('--train_file'); p.add_argument('--val_file')
    p.add_argument('--train_n', type=int, default=1000, help='training examples (sst2)')
    p.add_argument('--eval_n', type=int, default=1000, help='validation examples (rte: 277 = all, sst2: 872 = all)')
    p.add_argument('--epochs', type=float, default=2)
    p.add_argument('--bs', type=int, default=8)
    p.add_argument('--max_len', type=int, default=192)
    p.add_argument('--lr', type=float, default=None, help='lora: AdamW lr (2e-4); latent_sgd: step per unit vote, in ternary steps (0.01)')
    p.add_argument('--clip', type=float, default=1.0, help='per-layer gradient norm clip (latent_sgd)')
    p.add_argument('--lora_r', type=int, default=16)
    p.add_argument('--lora_alpha', type=float, default=32)
    p.add_argument('--tau', type=float, default=32)
    p.add_argument('--leak', type=float, default=0.0, help='counter: per-step stochastic decay of each counter toward 0 '
                   '(fading memory as in Bop, still int8); window ~ 1/leak steps')
    p.add_argument('--vote_scale', type=float, default=0.1, help='counter learning rate: votes are scaled before stochastic rounding')
    p.add_argument('--grad_ternary', type=int, default=0, help='round the output gradient to {-1,0,+1} x one scale per token (unbiased, stochastic) '
                   'before the weight gradient: simulates multiplication-free training')
    p.add_argument('--sr_lr', type=float, default=0.0, help='sr arm: expected notches moved per unit normalized vote; '
                   '0 = vote_scale / tau, the same expected drift as the counter (memoryless stochastic rounding, as in DQT)')
    p.add_argument('--fp16_matmul', type=int, default=0, help='GPU: run the ternary matmuls on fp16 tensor cores (ternary weights '
                   'are exact in fp16; gradients are rescaled first so they do not underflow)')
    p.add_argument('--kappa', type=float, default=0.5, help='counter init from the bf16 master weights (0 = zero)')
    p.add_argument('--bop_gamma', type=float, default=0.01, help='bop: moving-average rate of the accumulator')
    p.add_argument('--bop_thr', type=float, default=3.0, help='bop: threshold on the moving average (in normalized-vote units)')
    p.add_argument('--threads', type=int, default=0, help='0 = number of physical cores')
    p.add_argument('--eval_every', type=int, default=0)
    p.add_argument('--also_eval', default='', help='comma-separated tasks to score after training without training on them '
                   '(forgetting check), e.g. sst2,agnews')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--device', default='auto', help='auto | cpu | cuda')
    p.add_argument('--cache', type=int, default=-1, help='keep fp32 copies of the ternary weights (1 = faster on CPU, needs ~8 GB more); -1 = auto: on for CPU')
    p.add_argument('--max_steps', type=int, default=0, help='stop after this many steps (timing runs)')
    p.add_argument('--grad_ckpt', type=int, default=1, help='recompute each layer in backward instead of storing its activations (much less memory, ~30%% slower)')
    p.add_argument('--skip_eval', action='store_true', help='no evaluation (timing runs)')
    return p.parse_args()


def rss_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 2**30
    except Exception:
        return float('nan')


# ----------------------------------------------------------------------------- ternary layers
def act_quant(x):
    """Per-token absmax int8 (BitNet b1.58 activations); identity gradient (STE)."""
    s = 127.0 / x.detach().abs().amax(-1, keepdim=True).clamp(min=1e-5)
    q = (x.detach() * s).round().clamp(-128, 127) / s
    return x + (q - x.detach())


def ternarize(W):
    beta = W.abs().mean().clamp(min=1e-5)
    return (W / beta).round().clamp(-1, 1).to(torch.int8), beta


def tlinear(xq, layer):
    """xq @ W_eff^T, on fp16 tensor cores when --fp16_matmul (fp32 accumulation, fp32 result)."""
    if layer.args.fp16_matmul and xq.is_cuda:
        return F.linear(xq.half(), layer.weight_half()).float()
    return F.linear(xq, layer.weight_eff())


class FusedTernary(torch.autograd.Function):
    """y = xq @ W_eff^T. In backward the layer's weight gradient is used immediately to update the
    layer and then discarded, so the full model gradient never exists in memory."""
    @staticmethod
    def forward(ctx, xq, anchor, layer):
        ctx.layer = layer
        ctx.save_for_backward(xq)
        return tlinear(xq, layer)

    @staticmethod
    def backward(ctx, go):
        (xq,) = ctx.saved_tensors
        layer = ctx.layer
        fp16 = layer.args.fp16_matmul and go.is_cuda
        if fp16:                                          # per-token rescale: gradients are tiny and would underflow in fp16
            a = go.abs().amax(-1, keepdim=True).clamp(min=1e-30)
            gx = ((go / a).half() @ layer.weight_half()).float() * a
        else:
            gx = go @ layer.weight_eff()                  # uses the weights as they were in forward
        g = go.reshape(-1, go.shape[-1])
        if layer.args.grad_ternary:
            scale = g.abs().amax(-1, keepdim=True).clamp(min=1e-30)     # per token
            p = g.abs() / scale                                         # in [0, 1]
            g = torch.sign(g) * (torch.rand_like(p) < p).to(g.dtype) * scale   # E[g] unchanged
        x2 = xq.reshape(-1, xq.shape[-1])
        if fp16:                                          # one scale per tensor (it must factor out of the token sum)
            G, X = g.abs().amax().clamp(min=1e-30), x2.abs().amax().clamp(min=1e-30)
            gW = ((g / G).half().t() @ (x2 / X).half()).float() * (G * X)
        else:
            gW = g.t() @ x2
        layer.update(gW)
        return gx, torch.zeros(1, dtype=go.dtype, device=go.device), None


class FrozenTernary(torch.autograd.Function):
    """y = xq @ W_eff^T with frozen ternary weights: the weights are rebuilt in backward instead of
    being stored for it (storing them costs a full fp32 copy of the model)."""
    @staticmethod
    def forward(ctx, xq, layer):
        ctx.layer = layer
        return F.linear(xq, layer.weight_eff())

    @staticmethod
    def backward(ctx, go):
        return go @ ctx.layer.weight_eff(), None


class TernaryLinear(nn.Module):
    def __init__(self, lin, mode, args):
        super().__init__()
        W = lin.weight.data.float()
        self.mode, self.args = mode, args
        self.dout, self.din = W.shape
        self.flips = 0
        if mode == 'latent_sgd':
            self.register_buffer('W', W, persistent=False)  # fp32 master copy, updated in backward
        else:
            t, beta = ternarize(W)
            self.register_buffer('t', t)
            self.beta = float(beta)
            if mode == 'bop':
                self.register_buffer('m', torch.zeros_like(W, dtype=torch.float16))   # 16-bit moving average
                self.flipped = []
            if mode == 'sr':
                self.flipped = []
                self.t_init = t.clone()                      # CPU copy: net change measured at the end
            if mode == 'counter':
                r = (W / beta - t.float()).clamp(-0.5, 0.5)  # how close each weight was to flipping
                lim = max(1, args.tau - 1)
                c = (args.kappa * 2 * r * args.tau).round().clamp(-lim, lim).clamp(-127, 127)
                self.register_buffer('c', c.to(torch.int8))
                self.flipped = []
        if mode in ('counter', 'bop', 'sr', 'latent_sgd'):
            self.anchor = nn.Parameter(torch.zeros(1))      # lets gradients flow through frozen embeddings
        if mode == 'lora':
            r = args.lora_r
            self.A = nn.Parameter(torch.randn(r, self.din) / math.sqrt(self.din))
            self.B = nn.Parameter(torch.zeros(self.dout, r))
            self.scale = args.lora_alpha / r

    def weight_eff(self):
        if self.mode == 'latent_sgd':
            t, beta = ternarize(self.W)
            return t.float() * beta
        if getattr(self, 'Wc', None) is not None:
            return self.Wc
        return self.t.float() * self.beta

    @torch.no_grad()
    def weight_half(self):
        """fp16 weights (t * beta). Cached while the layer is in eval mode (generation calls each layer once per
        token); never cached in training, where the weights change."""
        if self.mode == 'latent_sgd':
            return self.weight_eff().half()
        Wh = getattr(self, '_Wh', None)
        if Wh is None:
            Wh = self.t.half() * self.beta
            if not self.training:
                self._Wh = Wh
        return Wh

    def train(self, mode=True):
        self._Wh = None
        return super().train(mode)

    @torch.no_grad()
    def build_cache(self):
        self.Wc = self.t.float() * self.beta

    def forward(self, x):
        xq = act_quant(x)
        if self.mode in ('counter', 'bop', 'sr', 'latent_sgd') and self.training and torch.is_grad_enabled():
            return FusedTernary.apply(xq, self.anchor, self)
        if self.mode == 'lora' and torch.is_grad_enabled() and xq.requires_grad:
            y = FrozenTernary.apply(xq, self)
        else:
            y = tlinear(xq, self)
        if self.mode == 'lora':
            y = y + (x @ self.A.t()) @ self.B.t() * self.scale
        return y

    @torch.no_grad()
    def update(self, gW):
        a = self.args
        self._Wh = None
        if self.mode == 'latent_sgd':
            # same normalized step as the counter, accumulated in fp32 instead of an int8 counter:
            # each weight moves lr * beta per unit vote (lr = fraction of one ternary step)
            t_old, beta = ternarize(self.W)
            s = gW / gW.pow(2).mean().sqrt().clamp(min=1e-30)
            self.W.add_(s, alpha=-a.lr * float(beta))
            self.flips += int((ternarize(self.W)[0] != t_old).sum())
            return
        if self.mode == 'bop':
            # ternary Bop (Helwegen et al. 2019): moving average of the normalized votes, one notch per step
            # while it stays past the threshold, no reset
            s = -gW / gW.pow(2).mean().sqrt().clamp(min=1e-30)
            m = self.m.float().mul_(1 - a.bop_gamma).add_(s, alpha=a.bop_gamma)
            t = self.t.float()
            nt = torch.where(m >= a.bop_thr, (t + 1).clamp(max=1), torch.where(m <= -a.bop_thr, (t - 1).clamp(min=-1), t))
            changed = nt != t
            k = int(changed.sum())
            if k:
                idx = changed.flatten().nonzero().squeeze(1)
                self.flipped.append(idx.to(torch.int32).cpu()); self.flips += k
                if getattr(self, 'Wc', None) is not None:
                    self.Wc.view(-1)[idx] = nt.flatten()[idx] * self.beta
            self.m.copy_(m.to(torch.float16)); self.t.copy_(nt.to(torch.int8))
            return
        if self.mode == 'sr':
            # stochastic rounding straight onto the ternary grid (DQT-style, no memory): move one notch with
            # probability |d|; same normalized vote as the counter, same expected drift by default
            d = -(a.sr_lr or a.vote_scale / a.tau) * gW / gW.pow(2).mean().sqrt().clamp(min=1e-30)
            move = torch.rand_like(d) < d.abs().clamp(max=1)
            t = self.t.float()
            nt = torch.where(move & (d > 0), (t + 1).clamp(max=1), torch.where(move & (d < 0), (t - 1).clamp(min=-1), t))
            changed = nt != t
            k = int(changed.sum())
            if k:
                idx = changed.flatten().nonzero().squeeze(1)
                self.flips += k                         # many moves: positions not logged, net change from t_init
                if getattr(self, 'Wc', None) is not None:
                    self.Wc.view(-1)[idx] = nt.flatten()[idx] * self.beta
                self.t.copy_(nt.to(torch.int8))
            return
        s = -a.vote_scale * gW / gW.pow(2).mean().sqrt().clamp(min=1e-30)
        cf = self.c.float()
        if a.leak:                                         # E[c] shrinks by (1 - leak) per step, stays an integer
            cf = cf - torch.sign(cf) * (torch.rand_like(cf) < (a.leak * cf.abs()).clamp(max=1)).float()
        cf = (cf + torch.floor(s + torch.rand_like(s))).clamp(-127, 127)
        t = self.t.float()
        up, dn = cf >= a.tau, cf <= -a.tau
        nt = torch.where(up, (t + 1).clamp(max=1), torch.where(dn, (t - 1).clamp(min=-1), t))
        cf = torch.where(up | dn, torch.zeros_like(cf), cf)
        changed = nt != t
        k = int(changed.sum())
        if k:
            idx = changed.flatten().nonzero().squeeze(1)
            self.flipped.append(idx.to(torch.int32).cpu())
            self.flips += k
            if getattr(self, 'Wc', None) is not None:
                self.Wc.view(-1)[idx] = nt.flatten()[idx] * self.beta
        self.c.copy_(cf.to(torch.int8))
        self.t.copy_(nt.to(torch.int8))

    @torch.no_grad()
    def merge_lora(self, rescale=False):
        """Fold the adapter into the ternary weights. rescale=False keeps the layer's scale beta and rounds
        t + delta/beta back to {-1, 0, 1} (a weight changes only where the adapter pushed it past a rounding
        boundary). rescale=True re-ternarizes from scratch, recomputing beta = mean|W| (changes every layer's scale)."""
        if not hasattr(self, 't0'):                        # keep the adapter low-rank (a full fp32 delta per layer = ~8 GB)
            self.t0, self.beta0 = self.t.cpu().clone(), self.beta
            self.Am, self.Bm = self.A.detach().clone(), self.B.detach().clone()
            self.mode = 'frozen'
            del self.A, self.B
        W = self.t0.to(self.t.device).float() * self.beta0 + self.scale * (self.Bm @ self.Am)
        if rescale:
            t, beta = ternarize(W)
        else:
            t, beta = (W / self.beta0).round().clamp(-1, 1).to(torch.int8), self.beta0
        self.t.copy_(t); self.beta = float(beta); self._Wh = None
        if getattr(self, 'Wc', None) is not None:
            self.build_cache()


def build(args):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(args.model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    tok.truncation_side = 'left'                         # keep the question and 'Answer:' at the end
    try:
        model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.bfloat16, low_cpu_mem_usage=True)
    except TypeError:                                     # older transformers
        model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
    model.config.use_cache = False
    mode = {'zeroshot': 'frozen'}.get(args.arm, args.arm)
    layers = []
    for name, mod in list(model.named_modules()):
        for cname, child in list(mod.named_children()):
            w = getattr(child, 'weight', None)
            if cname in TARGETS and isinstance(w, torch.Tensor) and w.dim() == 2:
                tl = TernaryLinear(child, mode, args)
                setattr(mod, cname, tl)
                layers.append(tl)
                del child
    assert layers, 'no q/k/v/o/gate/up/down projections found: check the model architecture'
    # transformers marks BitNet as a 'quantized' model and blocks .float()/.to(); its online-quantized
    # linear layers have just been replaced by ours, so the plain torch conversion is safe here
    nn.Module.float(model)                                # fp32 math; int8 buffers stay int8
    model.requires_grad_(False)
    for l in layers:
        for n in ('anchor', 'A', 'B'):
            if hasattr(l, n):
                getattr(l, n).requires_grad_(True)
    return model, tok, layers


# ----------------------------------------------------------------------------------------- data
DEV = torch.device('cpu')


def glue(name):
    from datasets import load_dataset
    try:
        return load_dataset('nyu-mll/glue', name)
    except Exception:
        return load_dataset('glue', name)


def cls_data(args):
    """Returns (train, val) as lists of (prompt, label index)."""
    from datasets import load_dataset
    if args.task == 'sst2':
        ds = glue('sst2'); tr, va = ds['train'], ds['validation']
        fmt = lambda r: (f"Review: {r['sentence'].strip()}\nSentiment:", r['label'])
    elif args.task == 'rte':
        ds = glue('rte'); tr, va = ds['train'], ds['validation']       # label 0 = entailment, 1 = not
        fmt = lambda r: (f"{r['sentence1'].strip()}\nQuestion: Does this imply that \"{r['sentence2'].strip()}\"? Yes or No?\nAnswer:", r['label'])
    else:
        ds = load_dataset('fancyzhx/ag_news'); tr, va = ds['train'], ds['test']
        fmt = lambda r: (f"Article: {r['text'].strip()}\nTopic (World, Sports, Business or Science):", r['label'])
    tr = tr.shuffle(seed=args.seed).select(range(min(args.train_n, len(tr))))
    va = va.shuffle(seed=1234).select(range(min(args.eval_n, len(va)))) if args.task == 'agnews' else va.select(range(min(args.eval_n, len(va))))
    return [fmt(r) for r in tr], [fmt(r) for r in va]


def cls_logits(model, tok, batch, label_ids, args):
    enc = tok([p for p, _ in batch], return_tensors='pt', padding=True, truncation=True, max_length=args.max_len)
    ids, am = enc['input_ids'].to(DEV), enc['attention_mask'].to(DEV)
    h = model.base_model(input_ids=ids, attention_mask=am).last_hidden_state
    last = am.sum(1) - 1
    h = h[torch.arange(h.shape[0], device=DEV), last]    # hidden state at the last prompt token
    W = model.get_output_embeddings().weight[label_ids]  # only the label words: skips the 128k-vocab head
    return h @ W.t(), int(ids.numel())                     # tokens processed, padding included


def text_data(args, tok):
    def chunks(path):
        ids = tok(open(path, encoding='utf-8').read(), add_special_tokens=False)['input_ids']
        L = args.max_len
        return [ids[i:i + L] for i in range(0, len(ids) - L + 1, L)]
    return chunks(args.train_file), chunks(args.val_file)


def text_loss(model, batch):
    x = torch.tensor(batch, device=DEV)
    return model(input_ids=x, labels=x).loss, x.numel()


def example_len(ex):
    if isinstance(ex, dict):
        return len(ex['ids'])
    return len(ex[0]) if isinstance(ex, tuple) else len(ex)


def make_batches(train, bs, rng):
    """Shuffle, then group examples of similar length into the same batch (less padding),
    then shuffle the order of the batches."""
    order = list(range(len(train))); rng.shuffle(order)
    batches = []
    pool = bs * 16
    for i in range(0, len(order), pool):
        chunk = sorted(order[i:i + pool], key=lambda j: example_len(train[j]))
        batches += [chunk[k:k + bs] for k in range(0, len(chunk), bs)]
    rng.shuffle(batches)
    return batches


@torch.no_grad()
def evaluate(model, tok, val, label_ids, args):
    model.eval()
    order = sorted(range(len(val)), key=lambda j: example_len(val[j]))
    if args.task != 'text':
        correct = 0
        for i in range(0, len(order), 16):
            b = [val[j] for j in order[i:i + 16]]
            lg, _ = cls_logits(model, tok, b, label_ids, args)
            correct += (lg.argmax(1).cpu() == torch.tensor([y for _, y in b])).sum().item()
        res = dict(accuracy=correct / len(val))
    else:
        tot = n = 0
        for i in range(0, len(val), args.bs):
            loss, k = text_loss(model, val[i:i + args.bs]); tot += loss.item() * k; n += k
        res = dict(val_loss=tot / n, ppl=math.exp(tot / n))
    model.train()
    return res


def eval_other_tasks(model, tok, args):
    """Score the model on other tasks it was not trained on (forgetting check)."""
    import copy
    out = {}
    for task in [t for t in args.also_eval.split(',') if t]:
        a2 = copy.copy(args); a2.task = task
        a2.eval_n = {'sst2': 872, 'rte': 277}.get(task, 500); a2.train_n = 1
        _, val = cls_data(a2)
        ids = [tok(w, add_special_tokens=False)['input_ids'][0] for w in TASKS[task]['labels']]
        out[task] = evaluate(model, tok, val, ids, a2)['accuracy']
        print(f'ALSO_EVAL {task}: accuracy {out[task]:.4f}', flush=True)
    return out


# ----------------------------------------------------------------------------------------- code
INSTR = 'Please provide a self-contained Python script that solves the following problem in a markdown code block:'
RESP = 'Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:\n```python\n'


def chat_prompt_text(tok, user):
    if getattr(tok, 'chat_template', None):
        return tok.apply_chat_template([{'role': 'user', 'content': user}], add_generation_prompt=True, tokenize=False)
    return f'User: {user}\nAssistant: '


def encode_chat(tok, user, assistant, max_len):
    p = tok(chat_prompt_text(tok, user), add_special_tokens=False)['input_ids']
    a = tok(assistant, add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    if len(p) + len(a) > max_len:
        return None                                       # drop instead of truncating the answer
    return dict(ids=p + a, n_prompt=len(p))


def benchmark_shingles():
    """10-word sequences and function names from every HumanEval+ and MBPP+ problem and reference solution."""
    from evalplus.data import get_human_eval_plus, get_mbpp_plus
    sh, names = set(), set()
    for ds in (get_human_eval_plus(), get_mbpp_plus()):
        for p in ds.values():
            w = re.findall(r'\w+', (p['prompt'] + '\n' + p.get('canonical_solution', '')).lower())
            sh.update(' '.join(w[i:i + 10]) for i in range(len(w) - 9))
            names.add(p['entry_point'])
    return sh, names


def code_data(args, tok):
    """Magicoder OSS-Instruct Python examples, minus anything overlapping the benchmarks."""
    from datasets import load_dataset
    ds = load_dataset('ise-uiuc/Magicoder-OSS-Instruct-75K', split='train').shuffle(seed=args.seed)
    sh, names = benchmark_shingles()
    keep, removed, too_long, scanned = [], 0, 0, 0
    need = args.train_n + args.code_dev
    for r in ds:
        sol = r['solution']
        if 'python' not in (r.get('lang') or '').lower() and '```python' not in sol:
            continue
        scanned += 1
        text = r['problem'] + '\n' + sol
        w = re.findall(r'\w+', text.lower())
        if any(' '.join(w[i:i + 10]) in sh for i in range(len(w) - 9)) or any(f'def {n}(' in text for n in names):
            removed += 1
            continue
        ex = encode_chat(tok, r['problem'], sol, args.max_len)
        if ex is None:
            too_long += 1
            continue
        keep.append(ex)
        if len(keep) >= need:
            break
    info = dict(python_scanned=scanned, removed_benchmark_overlap=removed, dropped_too_long=too_long, kept=len(keep))
    print(f'code data: {info}', flush=True)
    return keep[:args.train_n], keep[args.train_n:need], info


def _head_ce(hc, yc, W):
    return F.cross_entropy(hc @ W.t(), yc, reduction='sum')


def code_loss(model, batch, tok):
    """Next-token loss on the answer tokens only; the 128k-vocab head runs in recomputed chunks."""
    from torch.utils.checkpoint import checkpoint
    L = max(len(e['ids']) for e in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    am = torch.zeros_like(ids); lab = torch.full_like(ids, -100)
    for i, e in enumerate(batch):
        n = len(e['ids'])
        ids[i, :n] = torch.tensor(e['ids']); am[i, :n] = 1
        lab[i, e['n_prompt']:n] = ids[i, e['n_prompt']:n]
    ids, am, lab = ids.to(DEV), am.to(DEV), lab.to(DEV)
    h = model.base_model(input_ids=ids, attention_mask=am).last_hidden_state[:, :-1]
    y = lab[:, 1:]
    sel = y != -100
    hs, ys = h[sel], y[sel]
    W = model.get_output_embeddings().weight
    tot = 0.0
    for i in range(0, len(ys), 512):
        if torch.is_grad_enabled():
            tot = tot + checkpoint(_head_ce, hs[i:i + 512], ys[i:i + 512], W, use_reentrant=False)
        else:
            tot = tot + _head_ce(hs[i:i + 512], ys[i:i + 512], W)
    return tot / max(len(ys), 1), int(ids.numel())


@torch.no_grad()
def code_dev_loss(model, tok, dev, args):
    model.eval(); tot = n = 0.0
    for i in range(0, len(dev), args.bs):
        b = dev[i:i + args.bs]
        loss, _ = code_loss(model, b, tok)
        k = sum(len(e['ids']) - e['n_prompt'] for e in b)
        tot += loss.item() * k; n += k
    model.train()
    return tot / max(n, 1)


@torch.no_grad()
def code_eval(model, tok, args, tag):
    """Greedy generation for every HumanEval+ / MBPP+ problem, scored by EvalPlus (runs the code in subprocesses)."""
    import subprocess, sys
    from evalplus.data import get_human_eval_plus, get_mbpp_plus, write_jsonl
    model.eval(); tok.padding_side = 'left'; res = {}
    for bench in [b for b in args.code_eval.split(',') if b]:
        probs = get_human_eval_plus() if bench == 'humaneval' else get_mbpp_plus()
        items = list(probs.items())
        if args.code_eval_n:
            items = items[:args.code_eval_n]
        prompts = [chat_prompt_text(tok, f"{INSTR}\n```python\n{p['prompt'].strip()}\n```") + RESP for _, p in items]
        order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
        sols = [None] * len(items); t0 = time.time()
        for k in range(0, len(order), args.gen_bs):
            idx = order[k:k + args.gen_bs]
            enc = tok([prompts[i] for i in idx], return_tensors='pt', padding=True, add_special_tokens=False).to(DEV)
            out = model.generate(**enc, max_new_tokens=args.max_new, do_sample=False, use_cache=True,
                                 pad_token_id=tok.pad_token_id, stop_strings=['```'], tokenizer=tok)
            for j, i in enumerate(idx):
                text = tok.decode(out[j, enc['input_ids'].shape[1]:], skip_special_tokens=True)
                sols[i] = text.split('```')[0]
        path = os.path.join(args.out, f'{tag}_{bench}_samples.jsonl')
        write_jsonl(path, [dict(task_id=tid, solution=sol) for (tid, _), sol in zip(items, sols)])
        r = dict(generation_seconds=time.time() - t0, samples=path)
        if not args.code_eval_n:
            ev = subprocess.run([sys.executable, '-m', 'evalplus.evaluate', '--dataset', bench, '--samples', path],
                                capture_output=True, text=True)
            passes = re.findall(r'pass@1:\s*([0-9.]+)', ev.stdout)
            r.update(base=float(passes[0]) if passes else None, plus=float(passes[1]) if len(passes) > 1 else None)
            if not passes:
                r['evalplus_output'] = (ev.stdout + ev.stderr)[-1500:]
        else:
            r['example'] = sols[0][:300]
        res[bench] = r
        print(f'CODE_EVAL [{tag}] {bench}: ' + json.dumps({k: v for k, v in r.items() if k != 'samples'}), flush=True)
    tok.padding_side = 'right'; model.train()
    return res


def mem_gb():
    m = rss_gb()
    if DEV.type == 'cuda':
        return max(m if m == m else 0, torch.cuda.max_memory_allocated() / 2**30)
    return m


# ----------------------------------------------------------------------------------------- main
def main():
    global DEV
    args = get_args()
    os.makedirs(args.out, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    if args.lr is None:
        args.lr = {'lora': 2e-4, 'latent_sgd': 0.01}.get(args.arm, 0.0)
    if args.device == 'auto':
        args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    DEV = torch.device(args.device)
    torch.backends.cuda.matmul.allow_tf32 = False         # true fp32 on GPU: same math as the CPU runs
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False   # fp16 matmuls accumulate in fp32
    threads = args.threads or max(1, (os.cpu_count() or 2) // 2)
    torch.set_num_threads(threads)
    if args.cache == -1:
        args.cache = int(DEV.type == 'cpu')
    print(f'arm {args.arm} | device {DEV} | {threads} CPU threads | weight cache {bool(args.cache)} | '
          f'RAM in use {rss_gb():.1f} GB', flush=True)

    if args.arm == 'latent_adamw':
        n = 2.08e9
        print(f'Full fine-tuning with AdamW keeps, per ternary weight, an fp32 master copy (4 B), two Adam '
              f'moments (8 B) and a gradient (4 B): about {n * 16 / 2**30:.0f} GB for this model\'s ~2.1B '
              f'ternary weights, before activations. It does not fit in 24 GB of RAM or a 16 GB GPU; not run.')
        json.dump(dict(arm='latent_adamw', fits=False, est_gb=n * 16 / 2**30), open(os.path.join(args.out, 'result.json'), 'w'))
        return

    t0 = time.time()
    model, tok, layers = build(args)
    nn.Module.to(model, DEV)
    if args.cache and args.arm != 'latent_sgd':
        for l in layers:
            l.build_cache()
    if args.grad_ckpt and args.arm != 'zeroshot':
        # safe with in-backward updates: a decoder layer is recomputed before its own backward starts,
        # i.e. before any of its weights are updated
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
            print('gradient checkpointing on', flush=True)
        except Exception as e:
            print(f'gradient checkpointing unavailable ({type(e).__name__}: {e}); running without it', flush=True)
    n_w = sum(l.dout * l.din for l in layers)
    state = {'zeroshot': 1, 'counter': 2, 'bop': 3, 'sr': 1, 'lora': 1, 'latent_sgd': 4}[args.arm]
    print(f'loaded in {time.time() - t0:.0f}s | {len(layers)} ternary layers, {n_w / 1e9:.2f}B weights | '
          f'training state of the ternary layers ~{n_w * state / 2**30:.1f} GB | memory in use {mem_gb():.1f} GB', flush=True)

    code_info = None
    if args.task == 'code':
        train, val, code_info = code_data(args, tok); label_ids = None
    elif args.task != 'text':
        train, val = cls_data(args)
        label_ids = [tok(w, add_special_tokens=False)['input_ids'][0] for w in TASKS[args.task]['labels']]
        assert len(set(label_ids)) == len(label_ids), 'label words share their first token'
        counts = np.bincount([y for _, y in val], minlength=len(label_ids))
        print(f'task {args.task}: {len(train)} train / {len(val)} eval examples | majority-class accuracy '
              f'{counts.max() / counts.sum():.3f}', flush=True)
    elif args.task == 'text':
        train, val = text_data(args, tok); label_ids = None
    log_f = open(os.path.join(args.out, 'log.jsonl'), 'a')

    if args.arm == 'zeroshot' and args.task == 'code':
        res = dict(arm='zeroshot', task='code', data=code_info, eval=dict(dev_loss=code_dev_loss(model, tok, val, args)))
        print('EVAL dev loss', res['eval'], flush=True)
        res['code'] = code_eval(model, tok, args, 'zeroshot')
        if args.also_eval:
            res['also_eval'] = eval_other_tasks(model, tok, args)
        json.dump(res, open(os.path.join(args.out, 'result.json'), 'w'), indent=1)
        print('RESULT', json.dumps({k: v for k, v in res.items() if k != 'data'}), flush=True)
        return
    if args.arm == 'zeroshot':
        t1 = time.time(); res = evaluate(model, tok, val, label_ids, args)
        res.update(arm='zeroshot', device=str(DEV), eval_seconds=time.time() - t1, peak_mem_gb=mem_gb())
        if args.also_eval:
            res['also_eval'] = eval_other_tasks(model, tok, args)
        print('RESULT', json.dumps(res), flush=True)
        json.dump(res, open(os.path.join(args.out, 'result.json'), 'w'), indent=1)
        return

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=0.0) if args.arm == 'lora' else None
    steps = int(math.ceil(len(train) / args.bs) * args.epochs)
    if args.max_steps:
        steps = min(steps, args.max_steps)
    print(f'{steps} steps of {args.bs} examples', flush=True)
    model.train()
    peak, tok_count, t_train = mem_gb(), 0, time.time()
    rng = random.Random(args.seed)
    batches = []
    for step in range(steps):
        if not batches:
            batches = make_batches(train, args.bs, rng)
        batch = [train[i] for i in batches.pop()]
        if args.task == 'code':
            loss, k = code_loss(model, batch, tok)
        elif args.task != 'text':
            lg, k = cls_logits(model, tok, batch, label_ids, args)
            loss = F.cross_entropy(lg, torch.tensor([y for _, y in batch], device=DEV))
        else:
            loss, k = text_loss(model, batch)
        loss.backward()                                   # counter / latent_sgd update their layers here
        if opt:
            opt.step(); opt.zero_grad(set_to_none=True)
        else:
            for p in params:
                p.grad = None
        tok_count += k
        peak = max(peak, mem_gb())
        el = time.time() - t_train
        flips = sum(l.flips for l in layers) / n_w if args.arm in ('counter', 'bop', 'sr', 'latent_sgd') else None
        rec = dict(step=step + 1, loss=loss.item(), tok_s=tok_count / el, s_per_step=el / (step + 1),
                   elapsed=el, mem_gb=mem_gb(), flips=flips)
        log_f.write(json.dumps(rec) + '\n'); log_f.flush()
        if step < 3 or (step + 1) % 10 == 0 or step + 1 == steps:
            eta = el / (step + 1) * (steps - step - 1)
            print(f"step {step + 1}/{steps} loss {rec['loss']:.4f} | {rec['tok_s']:.1f} tok/s | "
                  f"{rec['s_per_step']:.1f} s/step | ETA {eta / 60:.0f} min | memory {rec['mem_gb']:.1f} GB"
                  + (f' | flipped {flips * 100:.4f}%' if flips is not None else ''), flush=True)
        if args.eval_every and (step + 1) % args.eval_every == 0 and step + 1 < steps and not args.skip_eval:
            r = evaluate(model, tok, val, label_ids, args)
            log_f.write(json.dumps(dict(step=step + 1, eval=r)) + '\n'); print('EVAL', step + 1, r, flush=True)

    train_s = time.time() - t_train
    res = dict(arm=args.arm, device=str(DEV), steps=steps, train_seconds=train_s, s_per_step=train_s / steps,
               tok_s=tok_count / train_s, peak_mem_gb=peak, args={k: v for k, v in vars(args).items()})
    if args.arm in ('counter', 'bop', 'sr', 'latent_sgd'):
        res['flips_total_frac'] = sum(l.flips for l in layers) / n_w   # flips counted every step (a weight can flip more than once)
    if args.skip_eval:
        print('RESULT', json.dumps({k: v for k, v in res.items() if k != 'args'}), flush=True)
        json.dump(res, open(os.path.join(args.out, 'result.json'), 'w'), indent=1)
        return
    if args.task == 'code':
        res['data'] = code_info
        res['eval'] = dict(dev_loss=code_dev_loss(model, tok, val, args))
        print('EVAL final dev loss', res['eval'], flush=True)
        res['code'] = code_eval(model, tok, args, 'adapter' if args.arm == 'lora' else 'final')
    else:
        res['eval'] = evaluate(model, tok, val, label_ids, args)
        print('EVAL final', res['eval'], flush=True)
    if args.also_eval and args.arm != 'lora':
        res['also_eval'] = eval_other_tasks(model, tok, args)

    if args.arm == 'lora':
        n_ad = sum(p.numel() for l in layers for p in (l.A, l.B))
        res['adapter_mb'] = n_ad * 4 / 2**20
        for l in layers:
            l.merge_lora(rescale=False)
        res['merge_changed_frac'] = sum(int((l.t.cpu() != l.t0).sum()) for l in layers) / n_w
        if DEV.type == 'cuda':
            torch.cuda.empty_cache()
        if args.task == 'code' and res['merge_changed_frac'] == 0:
            res['code_merged'] = 'identical to the zero-shot model (no weight changed), generation skipped'
            res['eval_merged_ternary'] = dict(dev_loss=code_dev_loss(model, tok, val, args))
        elif args.task == 'code':
            res['eval_merged_ternary'] = dict(dev_loss=code_dev_loss(model, tok, val, args))
            res['code_merged'] = code_eval(model, tok, args, 'merged')
        else:
            res['eval_merged_ternary'] = evaluate(model, tok, val, label_ids, args)
        if args.also_eval:
            res['also_eval'] = eval_other_tasks(model, tok, args)        # on the deployable merged model
        print(f"EVAL after merging LoRA into ternary weights (scale kept; {res['merge_changed_frac']*100:.4f}% of weights changed)",
              res['eval_merged_ternary'], flush=True)
        if args.task != 'code':
            for l in layers:
                l.merge_lora(rescale=True)
            res['eval_merged_ternary_rescaled'] = evaluate(model, tok, val, label_ids, args)
            print('EVAL after merging with re-ternarization (scale recomputed)', res['eval_merged_ternary_rescaled'], flush=True)
    if args.arm in ('counter', 'bop', 'sr'):
        patch, n_pos = {}, 0
        for i, l in enumerate(layers):
            if l.mode == 'sr':
                pos = (l.t.flatten().cpu() != l.t_init.flatten()).nonzero().squeeze(1)
                if not len(pos):
                    continue
            elif l.flipped:
                pos = torch.unique(torch.cat(l.flipped))
            else:
                continue
            patch[f'pos{i}'] = pos.numpy()
            patch[f'val{i}'] = l.t.flatten().cpu()[pos.long()].numpy()
            n_pos += len(pos)
        path = os.path.join(args.out, 'flip_patch.npz')
        np.savez_compressed(path, **patch)
        res.update(weights_changed=n_pos, weights_changed_frac=n_pos / n_w,
                   flip_patch_mb=os.path.getsize(path) / 2**20)
        print(f'{n_pos} weights changed ({n_pos / n_w * 100:.4f}%), flip patch {res["flip_patch_mb"]:.1f} MB', flush=True)

    print('RESULT', json.dumps({k: v for k, v in res.items() if k != 'args'}), flush=True)
    json.dump(res, open(os.path.join(args.out, 'result.json'), 'w'), indent=1)


if __name__ == '__main__':
    main()

## Download the model, the data and the benchmarks once

In [ ]:
from huggingface_hub import snapshot_download
from datasets import load_dataset
from evalplus.data import get_human_eval_plus, get_mbpp_plus
snapshot_download('microsoft/bitnet-b1.58-2B-4T-bf16')
load_dataset('ise-uiuc/Magicoder-OSS-Instruct-75K', split='train')
try: load_dataset('nyu-mll/glue', 'sst2')
except Exception: load_dataset('glue', 'sst2')
print(len(get_human_eval_plus()), 'HumanEval+ problems,', len(get_mbpp_plus()), 'MBPP+ problems')

## Quick check (~10 min): tiny SR, counter and LoRA runs (merge path included), generation on 4 problems

In [ ]:
import subprocess, sys, os, re
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
W = '/kaggle/working'
for name, extra in [('sr', ['--arm', 'sr', '--fp16_matmul', '1', '--grad_ternary', '1']),
                    ('counter', ['--arm', 'counter', '--fp16_matmul', '1', '--grad_ternary', '1', '--leak', '0.003']),
                    ('lora', ['--arm', 'lora', '--lr', '2e-4'])]:
    r = subprocess.run([sys.executable, f'{W}/bitnet_cpu_finetune.py', '--device', 'cuda', '--task', 'code', '--vote_scale', '0.03',
                        '--train_n', '24', '--code_dev', '8', '--epochs', '1', '--max_len', '768', '--bs', '8',
                        '--code_eval_n', '4', '--code_eval', 'humaneval', '--out', f'{W}/runs/check_{name}'] + extra,
                       capture_output=True, text=True)
    print(f'--- {name} ---'); print(r.stdout[-1500:]); print(r.stderr[-1500:])
    assert r.returncode == 0, f'{name}: fix the error above before running the queues'

## Run both queues in parallel

Progress lines every 5 minutes; full logs in `/kaggle/working/<run>.out`. Finished runs are skipped if you re-run the cell.

In [ ]:
import threading, time, json
COMMON = ['--device', 'cuda', '--task', 'code', '--train_n', '2000', '--code_dev', '200', '--epochs', '1',
          '--max_len', '768', '--bs', '8', '--also_eval', 'sst2']
SESSION = 'B'          # 'A' done (gate G1 passed on 18 Oct); 'B' = leak study + slower SR
C  = ['--vote_scale', '0.03', '--fp16_matmul', '1', '--grad_ternary', '1']
CT = ['--arm', 'counter'] + C
SR = ['--arm', 'sr'] + C
LO = ['--arm', 'lora', '--lr', '2e-4']
QUEUES = {
  'A': ([('code_gt_s2', CT + ['--seed', '2']), ('code_sr_s0', SR + ['--seed', '0']),
         ('code_lora_s0', LO + ['--seed', '0']), ('code_zeroshot', ['--arm', 'zeroshot'])],
        [('code_sr_s1', SR + ['--seed', '1']), ('code_sr_s2', SR + ['--seed', '2']),
         ('code_lora_s1', LO + ['--seed', '1']), ('code_gt_leak3_s1', CT + ['--leak', '0.003', '--seed', '1'])]),
  'B': ([('code_gt_leak3_s2', CT + ['--leak', '0.003', '--seed', '2']), ('code_gt_leak1_s0', CT + ['--leak', '0.001', '--seed', '0']),
         ('code_sr_low_s0', SR + ['--sr_lr', '0.0000938', '--seed', '0'])],
        [('code_gt_leak1_s1', CT + ['--leak', '0.001', '--seed', '1']), ('code_gt_leak1_s2', CT + ['--leak', '0.001', '--seed', '2']),
         ('code_sr_vlow_s0', SR + ['--sr_lr', '0.00000938', '--seed', '0'])])}
Q0, Q1 = QUEUES[SESSION]

def run_queue(gpu, queue):
    for name, a in queue:
        if os.path.exists(f'{W}/runs/{name}/result.json'):
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        with open(f'{W}/{name}.out', 'w') as f:
            subprocess.run([sys.executable, f'{W}/bitnet_cpu_finetune.py'] + COMMON + a + ['--out', f'{W}/runs/{name}'],
                           env=env, stdout=f, stderr=subprocess.STDOUT)

threads = [threading.Thread(target=run_queue, args=(0, Q0)), threading.Thread(target=run_queue, args=(1, Q1))]
t0 = time.time()
for t in threads: t.start()
while any(t.is_alive() for t in threads):
    time.sleep(300)
    for name, _ in Q0 + Q1:
        p = f'{W}/{name}.out'
        if os.path.exists(p) and not os.path.exists(f'{W}/runs/{name}/result.json'):
            lines = [l for l in open(p).read().splitlines() if l.strip()]
            if lines: print(f'[{(time.time() - t0) / 60:5.0f} min] {name}: {lines[-1][:170]}')
for t in threads: t.join()
print(f'all done in {(time.time() - t0) / 60:.0f} min')

## Results, 3-seed means and gate G1

In [ ]:
import json, glob, statistics as st
PREV = {  # earlier sessions (same settings), MBPP+ / HumanEval+ in %
  'gt':     [('s0', 36.0, 31.7), ('s1', 36.5, 31.1), ('s2', 34.4, 31.1)],
  'leak3':  [('s0', 36.8, 33.5), ('s1', 34.9, 36.6)],
  'sr':     [('s0', 5.3, 4.9), ('s1', 2.4, 4.3), ('s2', 5.3, 0.0)],
  'lora_adapter': [('s0', 36.2, 28.7), ('s1', 35.2, 29.3)],
}
ZS = (32.3, 38.4)
def load(name):
    p = f'{W}/runs/{name}/result.json'
    return json.load(open(p)) if os.path.exists(p) else None
def scores(c):
    if not c or isinstance(c, str): return None
    m, h = c.get('mbpp', {}).get('plus'), c.get('humaneval', {}).get('plus')
    return None if m is None else (100 * m, 100 * h if h is not None else float('nan'))
groups = {'gt': 'code_gt_s', 'leak3': 'code_gt_leak3_s', 'leak1': 'code_gt_leak1_s', 'sr': 'code_sr_s',
          'sr_low': 'code_sr_low_s', 'sr_vlow': 'code_sr_vlow_s', 'lora_adapter': 'code_lora_s'}
table = {}
for g, prefix in groups.items():
    rows = [(s_, m, h) for s_, m, h in PREV.get(g, [])]
    for p in sorted(glob.glob(f'{W}/runs/{prefix}[0-9]/result.json')):
        r = json.load(open(p)); sc = scores(r.get('code'))
        if sc: rows.append((p.split('/')[-2][-2:], *sc))
    if g == 'lora_adapter':
        merged = []
        for p in sorted(glob.glob(f'{W}/runs/code_lora_s[0-9]/result.json')):
            r = json.load(open(p)); cm = r.get('code_merged')
            merged.append((p.split('/')[-2][-2:], r.get('merge_changed_frac'), cm if isinstance(cm, str) else scores(cm)))
        table['lora_merged'] = merged
    table[g] = rows
z = load('code_zeroshot')
if z and scores(z.get('code')): ZS = scores(z['code'])
print(f'zero-shot: MBPP+ {ZS[0]:.1f} / HumanEval+ {ZS[1]:.1f}\n')
means = {}
for g in groups:
    rows = table[g]
    if not rows: print(f'{g:13s} no runs yet'); continue
    ms, hs = [r[1] for r in rows], [r[2] for r in rows]
    sd = st.stdev(ms) if len(ms) > 1 else float('nan')
    means[g] = (st.mean(ms), sd, len(ms))
    print(f'{g:13s} n={len(ms)}  MBPP+ ' + ' / '.join(f'{m:.1f}' for m in ms) + f'  -> mean {st.mean(ms):.1f} ± {sd:.1f}'
          + '   HumanEval+ ' + ' / '.join(f'{h:.1f}' for h in hs) + f'  -> mean {st.mean(hs):.1f}')
for s_, frac, sc in table.get('lora_merged', []):
    print(f'lora merged   {s_}: {frac * 100 if frac is not None else float("nan"):.4f}% of weights changed -> {sc}')
for name in sorted(glob.glob(f'{W}/runs/code_*/result.json')):
    r = json.load(open(name))
    if 'weights_changed_frac' in r:
        print(f"  {name.split('/')[-2]:22s} weights changed {r['weights_changed_frac'] * 100:.4f}% | {r['s_per_step']:.1f} s/step"
              + (f" | SST-2 {r['also_eval']['sst2']:.3f}" if 'also_eval' in r else ''))

print('\nGATE G1 (criteria fixed in advance):')
gt = means.get('gt')
c1 = gt is not None and gt[2] >= 3 and gt[0] >= 35.5
c2 = gt is not None and gt[0] >= ZS[0] + 3
lm = table.get('lora_merged', [])
c3 = (not lm) or all(   # no LoRA runs this session: session A measured 0% changed for both seeds
      (f == 0) or (isinstance(sc, tuple) and sc[0] <= ZS[0] + 1) for _, f, sc in lm)
sr = means.get('sr')
c4 = gt is not None and sr is not None and gt[0] - sr[0] >= 1.5
for ok, txt in [(c1, 'counter MBPP+ mean of 3 seeds >= 35.5'), (c2, 'counter >= zero-shot + 3'),
                (c3, 'merged LoRA <= zero-shot + 1 (0% changed counts as equal to zero-shot)'), (c4, 'counter beats SR by >= 1.5')]:
    print(f"  [{'PASS' if ok else 'FAIL / not yet'}] {txt}")
print('  => G1', 'PASSED' if all([c1, c2, c3, c4]) else 'not passed yet (or missing runs)')

## HumanEval-drop diagnostic: missing imports? Rescore with the prompt's imports added (CPU, ~10 min)

In [ ]:
import json, glob, re, subprocess, sys
from evalplus.data import get_human_eval_plus, write_jsonl
P = get_human_eval_plus()
imps = lambda p: [l for l in p.splitlines() if l.startswith(('import ', 'from '))]
for f in sorted(glob.glob(f'{W}/runs/code_*/*humaneval_samples.jsonl')):
    if f.endswith('_impfix.jsonl'): continue
    S = [json.loads(l) for l in open(f)]
    no_def = sum(f"def {P[s['task_id']]['entry_point']}" not in s['solution'] for s in S)
    no_imp = sum(any(l not in s['solution'] for l in imps(P[s['task_id']]['prompt'])) for s in S)
    out = f.replace('.jsonl', '_impfix.jsonl')
    write_jsonl(out, [dict(task_id=s['task_id'], solution='\n'.join(imps(P[s['task_id']]['prompt'])) + '\n' + s['solution']) for s in S])
    r = subprocess.run([sys.executable, '-m', 'evalplus.evaluate', '--dataset', 'humaneval', '--samples', out], capture_output=True, text=True)
    sc = re.findall(r'pass@1:\s*([0-9.]+)', r.stdout)
    print(f"{f.split('/runs/')[1]:50s} missing def {no_def:3d} | missing imports {no_imp:3d} | rescored HumanEval / HumanEval+: {sc}")